# data_pipeline_final

Build the frozen weekly SKU panel used in the thesis.

Outputs (CSV/MD only):
- data/processed/panel.csv
- data/processed/split_weeks.csv
- tables/dataset_stats.csv
- docs/dataset_card.md

Frozen decisions:
- unit: product-week (week starts on Monday via W-SUN)
- delivered orders only
- filters: n_items >= 10, n_unique_prices >= 2 (item level)
- filters: n_weeks >= 5, n_unique_weekly_prices >= 2 (panel level)
- price P: weekly average item price
- P0: per-SKU mean/median of P computed on train weeks only
- r = P / P0
- splits: train <= 2018-02-19, val <= 2018-05-07, test <= 2018-08-27

In [20]:
import pandas as pd
import numpy as np
from pathlib import Path

In [21]:
# Paths
ROOT = Path.cwd()
if not (ROOT / "data/olist_datasets").exists():
    if (ROOT.parent / "data/olist_datasets").exists():
        ROOT = ROOT.parent
    else:
        raise FileNotFoundError(
            f"Could not find data/olist_datasets relative to cwd: {ROOT}"
        )

DATA_DIR = ROOT / "data/olist_datasets"
OUT_DIR = ROOT / "data/processed"
TABLES_DIR = ROOT / "tables"
DOCS_DIR = ROOT / "docs"

OUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

# Filters
MIN_ITEMS = 10
MIN_UNIQUE_PRICES = 2
MIN_WEEKS = 5
MIN_UNIQUE_WEEKLY_PRICES = 2

# Split boundaries (week starts)
START_WEEK = "2016-10-03"
TRAIN_END = "2018-02-19"
VAL_END = "2018-05-07"
TEST_END = "2018-08-27"

# Baseline price (train only)
P0_METHOD = "mean"  # "mean" or "median"
if P0_METHOD not in {"mean", "median"}:
    raise ValueError("P0_METHOD must be 'mean' or 'median'.")

# Leakage-safe feature windows
ROLLING_WINDOW = 4

# Elasticity diagnostics thresholds (train only)
MIN_TRAIN_WEEKS_FOR_ELASTICITY = 5
MIN_TRAIN_PRICE_SPAN = 0.02
MIN_TRAIN_UNIQUE_PRICES = 2

# r clipping (train quantiles)
R_CLIP_Q_LOW = 0.01
R_CLIP_Q_HIGH = 0.99


In [22]:
# Load raw tables
orders = pd.read_csv(
    DATA_DIR / "olist_orders_dataset.csv",
    parse_dates=["order_purchase_timestamp"],
)
items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
reviews = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")
cat_map = pd.read_csv(DATA_DIR / "product_category_name_translation.csv")

print("orders", orders.shape)
print("items", items.shape)
print("products", products.shape)
print("reviews", reviews.shape)

orders (99441, 8)
items (112650, 7)
products (32951, 9)
reviews (99224, 7)


In [23]:
# Delivered-only orders + purchase week
orders = orders[orders["order_status"] == "delivered"].copy()
items = items.merge(
    orders[["order_id", "order_purchase_timestamp"]],
    on="order_id",
    how="inner",
)
items["purchase_week"] = items["order_purchase_timestamp"].dt.to_period("W-SUN").dt.start_time

print("delivered items", items.shape)

delivered items (110197, 9)


In [24]:
# Item-level product activity filter
prod_stats = items.groupby("product_id").agg(
    n_items=("order_id", "size"),
    n_orders=("order_id", "nunique"),
    n_unique_prices=("price", "nunique"),
)

eligible_products = prod_stats[
    (prod_stats["n_items"] >= MIN_ITEMS)
    & (prod_stats["n_unique_prices"] >= MIN_UNIQUE_PRICES)
].index

items = items[items["product_id"].isin(eligible_products)].copy()
print("items after item-level filter", items.shape)

items after item-level filter (39856, 9)


In [25]:
# Weekly SKU panel
weekly = (
    items.groupby(["product_id", "purchase_week"], as_index=False)
    .agg(
        demand=("order_id", "size"),
        n_orders=("order_id", "nunique"),
        avg_price=("price", "mean"),
        min_price=("price", "min"),
        max_price=("price", "max"),
        n_unique_prices=("price", "nunique"),
        price_std=("price", lambda x: x.std(ddof=0)),
        freight_mean=("freight_value", "mean"),
    )
)

weekly["price_range"] = weekly["max_price"] - weekly["min_price"]


In [26]:
# Panel-level filter (weeks + weekly price variation)
panel_stats = weekly.groupby("product_id").agg(
    n_weeks=("purchase_week", "nunique"),
    n_unique_weekly_prices=("avg_price", "nunique"),
)

eligible_panel_products = panel_stats[
    (panel_stats["n_weeks"] >= MIN_WEEKS)
    & (panel_stats["n_unique_weekly_prices"] >= MIN_UNIQUE_WEEKLY_PRICES)
].index

panel = weekly[weekly["product_id"].isin(eligible_panel_products)].copy()
print("panel rows", panel.shape)

panel rows (19153, 11)


In [27]:
# Product attributes + category translation
products = products.merge(
    cat_map,
    on="product_category_name",
    how="left",
)

product_cols = [
    "product_id",
    "product_category_name",
    "product_category_name_english",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
    "product_photos_qty",
    "product_name_lenght",
    "product_description_lenght",
]

panel = panel.merge(products[product_cols], on="product_id", how="left")
panel = panel.rename(
    columns={
        "product_name_lenght": "product_name_length",
        "product_description_lenght": "product_description_length",
    }
)

In [28]:
# Review aggregates (cumulative as-of-week)
item_reviews = items.merge(
    reviews[["order_id", "review_score", "review_creation_date"]],
    on="order_id",
    how="left",
)

item_reviews = item_reviews.dropna(subset=["review_score", "review_creation_date"]).copy()
item_reviews["review_week"] = pd.to_datetime(item_reviews["review_creation_date"]).dt.to_period("W-SUN").dt.start_time
item_reviews["low_review"] = (item_reviews["review_score"] <= 2).astype(int)

review_weekly = item_reviews.groupby(["product_id", "review_week"], as_index=False).agg(
    review_count=("review_score", "size"),
    review_sum=("review_score", "sum"),
    review_low_count=("low_review", "sum"),
)

review_weekly = review_weekly.sort_values(["product_id", "review_week"])
review_weekly["cum_review_count"] = review_weekly.groupby("product_id")["review_count"].cumsum()
review_weekly["cum_review_sum"] = review_weekly.groupby("product_id")["review_sum"].cumsum()
review_weekly["cum_low_count"] = review_weekly.groupby("product_id")["review_low_count"].cumsum()

review_weekly["sku_review_mean"] = review_weekly["cum_review_sum"] / review_weekly["cum_review_count"]
review_weekly["sku_share_low"] = review_weekly["cum_low_count"] / review_weekly["cum_review_count"]

review_cum = review_weekly[
    ["product_id", "review_week", "cum_review_count", "sku_review_mean", "sku_share_low"]
].rename(columns={"cum_review_count": "sku_review_count"})

panel = panel.merge(
    review_cum,
    left_on=["product_id", "purchase_week"],
    right_on=["product_id", "review_week"],
    how="left",
)
panel = panel.drop(columns=["review_week"])

panel = panel.sort_values(["product_id", "purchase_week"]).reset_index(drop=True)
panel[["sku_review_count", "sku_review_mean", "sku_share_low"]] = (
    panel.groupby("product_id")[["sku_review_count", "sku_review_mean", "sku_share_low"]].ffill()
)
panel["sku_review_count"] = panel["sku_review_count"].fillna(0).astype(int)


In [29]:
# Time features, split, baseline price (P0), relative price (r)
panel = panel.rename(columns={"purchase_week": "week"})
panel = panel.sort_values(["product_id", "week"]).reset_index(drop=True)

start_week = pd.Timestamp(START_WEEK)
train_end = pd.Timestamp(TRAIN_END)
val_end = pd.Timestamp(VAL_END)
test_end = pd.Timestamp(TEST_END)

panel = panel[(panel["week"] >= start_week) & (panel["week"] <= test_end)].copy()

panel["year"] = panel["week"].dt.year
panel["month"] = panel["week"].dt.month
panel["weekofyear"] = panel["week"].dt.isocalendar().week.astype(int)
panel["week_sin"] = np.sin(2 * np.pi * panel["weekofyear"] / 52.0)
panel["week_cos"] = np.cos(2 * np.pi * panel["weekofyear"] / 52.0)

panel["split"] = np.where(
    panel["week"] <= train_end,
    "train",
    np.where(panel["week"] <= val_end, "val", "test"),
)

panel["weeks_since_last_sale"] = (
    panel.groupby("product_id")["week"].diff().dt.days.div(7)
)

panel["demand_lag_1"] = panel.groupby("product_id")["demand"].shift(1)
panel["demand_lag_2"] = panel.groupby("product_id")["demand"].shift(2)
panel["demand_roll_4"] = (
    panel.groupby("product_id")["demand"]
    .apply(lambda s: s.shift(1).rolling(window=ROLLING_WINDOW, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)

panel["price_lag_1"] = panel.groupby("product_id")["avg_price"].shift(1)
panel["price_roll_4"] = (
    panel.groupby("product_id")["avg_price"]
    .apply(lambda s: s.shift(1).rolling(window=ROLLING_WINDOW, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)

train_mask = panel["split"] == "train"
if P0_METHOD == "mean":
    p0 = panel.loc[train_mask].groupby("product_id")["avg_price"].mean()
else:
    p0 = panel.loc[train_mask].groupby("product_id")["avg_price"].median()

panel = panel.merge(p0.rename("P0"), on="product_id", how="left")
panel = panel[panel["P0"].notna()].copy()

panel["r"] = panel["avg_price"] / panel["P0"]

train_mask = panel["split"] == "train"
r_clip_low = panel.loc[train_mask, "r"].quantile(R_CLIP_Q_LOW)
r_clip_high = panel.loc[train_mask, "r"].quantile(R_CLIP_Q_HIGH)
panel["r_clipped"] = panel["r"].clip(lower=r_clip_low, upper=r_clip_high)

panel["y"] = np.log1p(panel["demand"])

train_stats = panel.loc[train_mask].groupby("product_id").agg(
    train_weeks=("week", "nunique"),
    train_price_min=("r", "min"),
    train_price_max=("r", "max"),
    train_unique_prices=("avg_price", "nunique"),
)
train_stats["train_price_span"] = (
    train_stats["train_price_max"] - train_stats["train_price_min"]
)

panel = panel.merge(
    train_stats[["train_weeks", "train_price_span", "train_unique_prices"]],
    on="product_id",
    how="left",
)

panel["usable_for_elasticity"] = (
    (panel["train_weeks"] >= MIN_TRAIN_WEEKS_FOR_ELASTICITY)
    & (panel["train_price_span"] >= MIN_TRAIN_PRICE_SPAN)
    & (panel["train_unique_prices"] >= MIN_TRAIN_UNIQUE_PRICES)
).astype(int)


In [30]:
# Save panel and split weeks
panel_out = panel.copy()
panel_out["week"] = panel_out["week"].dt.strftime("%Y-%m-%d")

panel_out.to_csv(OUT_DIR / "panel.csv", index=False)

split_weeks = panel[["week", "split"]].drop_duplicates().sort_values("week")
split_weeks["week"] = split_weeks["week"].dt.strftime("%Y-%m-%d")
split_weeks.to_csv(OUT_DIR / "split_weeks.csv", index=False)

print("saved panel rows", len(panel_out))
print("saved split weeks", len(split_weeks))

saved panel rows 17970
saved split weeks 89


In [31]:
# Dataset statistics table
usable_products = int(
    train_stats[
        (train_stats["train_weeks"] >= MIN_TRAIN_WEEKS_FOR_ELASTICITY)
        & (train_stats["train_price_span"] >= MIN_TRAIN_PRICE_SPAN)
        & (train_stats["train_unique_prices"] >= MIN_TRAIN_UNIQUE_PRICES)
    ].shape[0]
)

stats = {
    "n_products": int(panel["product_id"].nunique()),
    "n_rows": int(len(panel)),
    "n_weeks": int(panel["week"].nunique()),
    "date_min": panel["week"].min().date().isoformat(),
    "date_max": panel["week"].max().date().isoformat(),
    "demand_mean": float(panel["demand"].mean()),
    "demand_median": float(panel["demand"].median()),
    "demand_max": int(panel["demand"].max()),
    "avg_price_mean": float(panel["avg_price"].mean()),
    "avg_price_median": float(panel["avg_price"].median()),
    "avg_price_min": float(panel["avg_price"].min()),
    "avg_price_max": float(panel["avg_price"].max()),
    "r_p05": float(panel["r"].quantile(0.05)),
    "r_p95": float(panel["r"].quantile(0.95)),
    "r_min": float(panel["r"].min()),
    "r_max": float(panel["r"].max()),
    "r_clip_low": float(r_clip_low),
    "r_clip_high": float(r_clip_high),
    "train_weeks_median": float(train_stats["train_weeks"].median()),
    "train_price_span_median": float(train_stats["train_price_span"].median()),
    "usable_for_elasticity_products": usable_products,
}

stats_df = pd.DataFrame([{"metric": k, "value": v} for k, v in stats.items()])
stats_df.to_csv(TABLES_DIR / "dataset_stats.csv", index=False)

stats_df.head()


,metric,value
0,n_products,1218
1,n_rows,17970
2,n_weeks,89
3,date_min,2016-10-03
4,date_max,2018-08-27


In [33]:
# Dataset card (draft)
card_lines = [
    "# Dataset card: Olist weekly SKU panel (v0)",
    "",
    "## Summary",
    "- Source: Olist Brazilian e-commerce dataset (Kaggle)",
    f"- Time span: {stats['date_min']} to {stats['date_max']}",
    "- Unit of analysis: product-week",
    f"- Filters: delivered only; n_items >= {MIN_ITEMS}; n_unique_prices >= {MIN_UNIQUE_PRICES}; n_weeks >= {MIN_WEEKS}; n_unique_weekly_prices >= {MIN_UNIQUE_WEEKLY_PRICES}",
    f"- Final size: {stats['n_products']} products, {stats['n_rows']} product-week rows",
    "",
    "## Target and price",
    "- Target: y = log1p(demand), where demand is weekly units sold",
    "- Price P: weekly average item price",
    f"- Baseline price P0: {P0_METHOD} of P computed on train weeks only",
    "- Relative price: r = P / P0",
    f"- r_clipped: r clipped to train quantiles [{R_CLIP_Q_LOW}, {R_CLIP_Q_HIGH}]",
    "",
    "## Features",
    "- Time features: year, month, weekofyear, week_sin, week_cos",
    f"- Leakage-safe lags/rolls (window={ROLLING_WINDOW}): demand_lag_1/2, demand_roll_4, price_lag_1, price_roll_4",
    "- Gap feature: weeks_since_last_sale",
    "- Price dispersion (weekly): price_std, price_range",
    "- Reviews (cumulative-as-of-week): sku_review_count, sku_review_mean, sku_share_low",
    "- Diagnostics: train_weeks, train_price_span, train_unique_prices, usable_for_elasticity",
    "",
    "## Splits",
    f"- Train: week <= {TRAIN_END}",
    f"- Validation: {TRAIN_END} < week <= {VAL_END}",
    f"- Test: {VAL_END} < week <= {TEST_END}",
    "",
    "## Notes",
    "- The panel is not a full product-week grid; weeks with zero sales are not imputed.",
    "- Review aggregates are cumulative; before the first review they are missing (count=0, mean/share NaN).",
    "- Price is observational and may be endogenous; elasticities are associational.",
]

card = "\n".join(card_lines) + "\n"
(DOCS_DIR / "dataset_card.md").write_text(card, encoding="utf-8")
print("wrote", DOCS_DIR / "dataset_card.md")


wrote ../docs/dataset_card.md
